<a href="https://colab.research.google.com/github/Zeldano118/QPon_NLP_PBA/blob/main/notebooks/01_Scraping_Tokenisasi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 1: QPon Review Scraping + Introduction to NLP & Tokenization
## App Information
- **App Name:** QPon
- **Google Play:** [QPon on Play
Store](https://play.google.com/store/apps/details?id=com.qpon.platform)
- **App ID:** `com.qpon.platform`
## Author
- **Name:** Zeldano Shan Oeffie
- **NRP:** 5026231118
- **Course:** Natural Language Processing (PBA)
- **Lecturer:** Irmasari Hafidz
## Objectives
1. Scrape all QPon user reviews from Google Play Store
2. Explore core NLP concepts: tokenization, corpus, and the NLP pipeline
3. Apply tokenization on the Gutenberg corpus (English) and IndoNLU SMSA dataset (Indonesian)
4. Run tokenization, stemming, and stopword removal on actual QPon reviews
5. Store raw data for full preprocessing in Week 2
## Process
1. **Scraping:** Collect all QPon reviews via `google_play_scraper` (`lang='id'`)
2. **English Tokenization:** NLTK Gutenberg corpus — classic literature text
3. **Indonesian Tokenization:** IndoNLU SMSA from Hugging Face — social media text
4. **QPon Tokenization:** Tokenization + stemming + stopword removal on scraped reviews
## References
- Jurafsky & Martin, *Speech and Language Processing* (3rd ed., 2023)
- Cahyawijaya et al., *IndoNLU* (ACL 2020)
- Bird et al., *NLP with Python* (NLTK Book, 2009)


In [15]:
# ============================================
# LIBRARY INSTALLATION
# ============================================
!pip install google-play-scraper PySastrawi nltk datasets -q

In [16]:
# ============================================
# IMPORT LIBRARIES
# ============================================
import pandas as pd
import numpy as np
from google_play_scraper import reviews_all, Sort
from google_play_scraper import app as app_info
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from datasets import load_dataset
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from collections import Counter
import re
import time
import os
# Download NLTK resources
nltk.download('gutenberg')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
print('All libraries imported successfully.')

All libraries imported successfully.


[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


---
## Section A: Scraping QPon Reviews from Google Play
Collecting **all** available QPon reviews from Google Play Store using `google_play_scraper`.
- **App ID:** `com.qpon.platform`
- **Language:** Indonesian (`lang='id'`)
- **Method:** `reviews_all()` — retrieves every review, not just the first 200
- **Sort:** Newest first (`Sort.NEWEST`)

In [17]:
# ============================================
# QPON APP INFORMATION
# ============================================
# Pull general metadata from the Play Store listing
qpon_info = app_info('com.qpon.platform', lang='id', country='id')
print('=== QPON APP INFO ===')
print(f"Name : {qpon_info['title']}")
print(f"Developer : {qpon_info['developer']}")
print(f"Rating : {qpon_info['score']:.2f}")
print(f"Total Reviews : {qpon_info['reviews']:,}")
print(f"Installs : {qpon_info['installs']}")
print(f"Genre : {qpon_info['genre']}")
print(f"Version : {qpon_info['version']}")

=== QPON APP INFO ===
Name : Qpon-Selalu Ada Diskon
Developer : QPON DIGITAL SINGAPORE
Rating : 3.52
Total Reviews : 5,193
Installs : 10.000.000+
Genre : Wisata
Version : 2.23.0.0


In [18]:
# ============================================
# SCRAPE ALL QPON REVIEWS
# ============================================
# reviews_all() fetches every available review
# May take a few minutes depending on volume
print('Starting QPon review scraping...')
print('(This may take a few minutes.)')
print()
start_time = time.time()
try:
 qpon_reviews = reviews_all(
 'com.qpon.platform', # QPon App ID
 lang='id', # Indonesian
 country='id', # Indonesia region
 sort=Sort.NEWEST, # newest first
 sleep_milliseconds=100 # 100 ms delay between requests
 )
 elapsed = time.time() - start_time
 print(f'Done in {elapsed:.1f}s \u2014 {len(qpon_reviews):,} reviews scraped.')
except Exception as e:
 print(f'Scraping error: {e}')
 qpon_reviews = []


Starting QPon review scraping...
(This may take a few minutes.)

Done in 2.6s — 4,641 reviews scraped.


In [19]:
# ============================================
# CONVERT TO DATAFRAME & SAVE
# ============================================
df_qpon = pd.DataFrame(qpon_reviews)
print('=== DATASET OVERVIEW ===')
print(f'Reviews : {df_qpon.shape[0]:,}')
print(f'Columns : {df_qpon.shape[1]}')
print(f'Fields : {df_qpon.columns.tolist()}')
print()
print('=== DATA TYPES ===')
print(df_qpon.dtypes)
print()
df_qpon.to_csv('qpon_reviews_raw.csv', index=False, encoding='utf-8')
print(f'Saved to qpon_reviews_raw.csv')
print(f'File size: {os.path.getsize("qpon_reviews_raw.csv") / 1024:.1f} KB')

=== DATASET OVERVIEW ===
Reviews : 4,641
Columns : 11
Fields : ['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion']

=== DATA TYPES ===
reviewId                        object
userName                        object
userImage                       object
content                         object
score                            int64
thumbsUpCount                    int64
reviewCreatedVersion            object
at                      datetime64[ns]
replyContent                    object
repliedAt               datetime64[ns]
appVersion                      object
dtype: object

Saved to qpon_reviews_raw.csv
File size: 2130.0 KB


In [20]:
# ============================================
# INITIAL EXPLORATION
# ============================================
print('=== FIRST 5 REVIEWS ===')
for i, row in df_qpon.head().iterrows():
 print(f"\n--- Review {i+1} ---")
 print(f"User : {row.get('userName', 'N/A')}")
 print(f"Rating : {'\u2B50' * row.get('score', 0)} ({row.get('score', 'N/A')}/5)")
 print(f"Date : {row.get('at', 'N/A')}")
 print(f"Review : {row.get('content', 'N/A')}")
print()
print('=== RATING DISTRIBUTION ===')
rating_dist = df_qpon['score'].value_counts().sort_index()
for rating, count in rating_dist.items():
 pct = count / len(df_qpon) * 100
 bar = '\u2588' * int(pct / 2)
 print(f" {'\u2B50'*rating} ({rating}): {count:>6,} reviews ({pct:5.1f}%) {bar}")
print()
print(f'Average rating: {df_qpon["score"].mean():.2f}')

=== FIRST 5 REVIEWS ===

--- Review 1 ---
User : Pengguna Google
Rating : ⭐⭐⭐⭐⭐ (5/5)
Date : 2026-03-03 01:42:30
Review : diskon ya gede

--- Review 2 ---
User : Pengguna Google
Rating : ⭐ (1/5)
Date : 2026-03-02 14:38:22
Review : setiap mau pesen tulisannya terlalu banyak peminat

--- Review 3 ---
User : Pengguna Google
Rating : ⭐⭐⭐⭐⭐ (5/5)
Date : 2026-03-02 11:18:14
Review : promonya besar-besaran, customers servicenya bener-bener membantu dan melayani dengan baik ketika ada kendala. sukses terus yaa qpon.

--- Review 4 ---
User : Pengguna Google
Rating : ⭐⭐⭐ (3/5)
Date : 2026-03-02 08:07:01
Review : br mau login, malah gbisa nih.. ket nya akun terdeteksi tdk normal

--- Review 5 ---
User : Pengguna Google
Rating : ⭐ (1/5)
Date : 2026-03-02 05:53:45
Review : apk fomo doang bisa sekarang susah banyak promo tapi gak bisa di pake buang buang duit doang.

=== RATING DISTRIBUTION ===
 ⭐ (1):  2,032 reviews ( 43.8%) █████████████████████
 ⭐⭐ (2):    267 reviews (  5.8%) ██
 ⭐⭐⭐ (3):    170

In [21]:
# ============================================
# SAMPLE REVIEWS BY RATING
# ============================================
for rating in range(1, 6):
    subset = df_qpon[df_qpon['score'] == rating]
    if len(subset) > 0:
        sample = subset['content'].dropna().head(2).tolist()
        print(f"\n{'='*60}")
        print(f"RATING {'\u2B50'*rating} ({rating}/5) \u2014 {len(subset):,} reviews")
        print(f"{'='*60}")
        for j, text in enumerate(sample):
            display_text = str(text)[:200] + '...' if len(str(text)) > 200 else str(text)
            print(f" Sample {j+1}: {display_text}")



RATING ⭐ (1/5) — 2,032 reviews
 Sample 1: setiap mau pesen tulisannya terlalu banyak peminat
 Sample 2: apk fomo doang bisa sekarang susah banyak promo tapi gak bisa di pake buang buang duit doang.

RATING ⭐⭐ (2/5) — 267 reviews
 Sample 1: duit gua kgk blik anjir salah beli juga minta tanggung jawab malah gk jelas balesan nya
 Sample 2: tolong buat menu riwayat transaksinya dicantumkan nama outlet juga di bagian depannya , sehingga ga perlu dibuka dulu satu2 dan klik detail pesanan.

RATING ⭐⭐⭐ (3/5) — 170 reviews
 Sample 1: br mau login, malah gbisa nih.. ket nya akun terdeteksi tdk normal
 Sample 2: karena pas flash sale yang di liat hanya brand aja alhasil gak bisa kepake dan jeleknya GAK ADA OPTION REFUND . saya di bandung salah beli voucher malah yang jakarta .. tolong dong qpon kasih kebijaka...

RATING ⭐⭐⭐⭐ (4/5) — 130 reviews
 Sample 1: saeee
 Sample 2: mantap diskon nya

RATING ⭐⭐⭐⭐⭐ (5/5) — 2,042 reviews
 Sample 1: diskon ya gede
 Sample 2: promonya besar-besaran, customers 

---
## Section B: English Tokenization (Gutenberg Corpus)
Tokenizing a passage from Jane Austen's *Emma* via NLTK's Gutenberg corpus to demonstrate
basic tokenization on formal English text.
**Ref:** Bird et al., *NLP with Python* (2009), Ch. 1

In [22]:
# ============================================
# ENGLISH TOKENIZATION (GUTENBERG)
# ============================================
# Jane Austen's Emma \u2014 first 1,000 characters
text_en = nltk.corpus.gutenberg.raw('austen-emma.txt')[:1000]
print('=== ORIGINAL TEXT (BEFORE) ===')
print(text_en[:500])
print()
tokens_en = word_tokenize(text_en)
print('=== AFTER TOKENIZATION ===')
print('First 20 tokens:', tokens_en[:20])
print()
print('=== STATS ===')
print(f'Total tokens : {len(tokens_en)}')
print(f'Unique tokens : {len(set(tokens_en))}')
print(f'Text length : {len(text_en)} chars')

=== ORIGINAL TEXT (BEFORE) ===
[Emma by Jane Austen 1816]

VOLUME I

CHAPTER I


Emma Woodhouse, handsome, clever, and rich, with a comfortable home
and happy disposition, seemed to unite some of the best blessings
of existence; and had lived nearly twenty-one years in the world
with very little to distress or vex her.

She was the youngest of the two daughters of a most affectionate,
indulgent father; and had, in consequence of her sister's marriage,
been mistress of his house from a very early period.  Her mother
had died t

=== AFTER TOKENIZATION ===
First 20 tokens: ['[', 'Emma', 'by', 'Jane', 'Austen', '1816', ']', 'VOLUME', 'I', 'CHAPTER', 'I', 'Emma', 'Woodhouse', ',', 'handsome', ',', 'clever', ',', 'and', 'rich']

=== STATS ===
Total tokens : 198
Unique tokens : 114
Text length : 1000 chars


---
## Section C: Indonesian Tokenization (IndoNLU SMSA)
Using the **IndoNLU SMSA** dataset from Hugging Face \u2014 Indonesian social-media text with
sentiment labels. Good contrast to the formal English text above, since user-generated
Indonesian content tends to be shorter, more informal, and full of slang.
**Ref:** Cahyawijaya et al., *IndoNLU* (ACL 2020).

In [23]:
# ============================================
# INDONESIAN TOKENIZATION (INDONLU SMSA)
# ============================================
# Dataset: kornwtp/indonlu-smsa (Hugging Face)
# Indonesian social media text + sentiment labels
dataset = load_dataset('kornwtp/indonlu-smsa', split='train')
df_indo = dataset.to_pandas()
print(f'Dataset size: {len(df_indo)} rows')
print(f'Columns: {list(df_indo.columns)}')
print()
text_col = 'text' if 'text' in df_indo.columns else df_indo.columns[0]
label_col = 'label' if 'label' in df_indo.columns else df_indo.columns[1]
sample_texts_indo = df_indo[text_col][:3].tolist()
sample_labels = df_indo[label_col][:3].tolist()
stemmer = StemmerFactory().create_stemmer()
for i, text in enumerate(sample_texts_indo):
    print(f'\nSample {i+1} (label: {sample_labels[i]}):')
    print(f'Original : {text}')
    tokens = word_tokenize(str(text))
    print(f'Tokens : {tokens}')
    print(f'Count : {len(tokens)} total, {len(set(tokens))} unique')
    stemmed_text = stemmer.stem(str(text))
    stemmed_tokens = word_tokenize(stemmed_text)
    print(f'Stemmed : {stemmed_tokens}')
    print(f'Count : {len(stemmed_tokens)}')
    with open(f'tokens_sample_{i+1}.txt', 'w', encoding='utf-8') as f:
        f.write('\n'.join(tokens))
    print(f'Saved to tokens_sample_{i+1}.txt')
all_tokens_indo = []
for text in sample_texts_indo:
    all_tokens_indo.extend(word_tokenize(str(text).lower()))
word_freq_indo = Counter(all_tokens_indo).most_common(10)
print(f'\nTop 10 tokens: {word_freq_indo}')

Dataset size: 11000 rows
Columns: ['texts', 'labels']


Sample 1 (label: 0):
Original : warung ini dimiliki oleh pengusaha pabrik tahu yang sudah puluhan tahun terkenal membuat tahu putih di bandung . tahu berkualitas , dipadu keahlian memasak , dipadu kretivitas , jadilah warung yang menyajikan menu utama berbahan tahu , ditambah menu umum lain seperti ayam . semuanya selera indonesia . harga cukup terjangkau . jangan lewatkan tahu bletoka nya , tidak kalah dengan yang asli dari tegal !
Tokens : ['warung', 'ini', 'dimiliki', 'oleh', 'pengusaha', 'pabrik', 'tahu', 'yang', 'sudah', 'puluhan', 'tahun', 'terkenal', 'membuat', 'tahu', 'putih', 'di', 'bandung', '.', 'tahu', 'berkualitas', ',', 'dipadu', 'keahlian', 'memasak', ',', 'dipadu', 'kretivitas', ',', 'jadilah', 'warung', 'yang', 'menyajikan', 'menu', 'utama', 'berbahan', 'tahu', ',', 'ditambah', 'menu', 'umum', 'lain', 'seperti', 'ayam', '.', 'semuanya', 'selera', 'indonesia', '.', 'harga', 'cukup', 'terjangkau', '.', 'jangan', 'le

---
## Section D: Tokenization on QPon Reviews
Now applying the same techniques to the actual QPon data scraped in Section A \u2014
bridging the textbook examples with real project data.

In [24]:
# ============================================
# QPON REVIEW TOKENIZATION
# ============================================
sample_reviews = df_qpon['content'].dropna().head(5).tolist()
print('=== TOKENIZATION — BEFORE vs AFTER ===')
print()
for i, review in enumerate(sample_reviews):
    print(f'--- QPon Review #{i+1} ---')
    print(f'BEFORE : {review}')
    tokens = word_tokenize(str(review))
    print(f'AFTER : {tokens}')
    print(f'Tokens : {len(tokens)}')
    print()
all_tokens_qpon = []
for review in df_qpon['content'].dropna():
    all_tokens_qpon.extend(word_tokenize(str(review).lower()))
print(f'Total tokens across all QPon reviews: {len(all_tokens_qpon):,}')
print(f'Unique tokens: {len(set(all_tokens_qpon)):,}')
print()
freq_qpon = Counter(all_tokens_qpon)
print('=== 15 MOST FREQUENT TOKENS ===')
for word, count in freq_qpon.most_common(15):
    print(f' {word:20s} : {count:>6,}')

=== TOKENIZATION — BEFORE vs AFTER ===

--- QPon Review #1 ---
BEFORE : diskon ya gede
AFTER : ['diskon', 'ya', 'gede']
Tokens : 3

--- QPon Review #2 ---
BEFORE : setiap mau pesen tulisannya terlalu banyak peminat
AFTER : ['setiap', 'mau', 'pesen', 'tulisannya', 'terlalu', 'banyak', 'peminat']
Tokens : 7

--- QPon Review #3 ---
BEFORE : promonya besar-besaran, customers servicenya bener-bener membantu dan melayani dengan baik ketika ada kendala. sukses terus yaa qpon.
AFTER : ['promonya', 'besar-besaran', ',', 'customers', 'servicenya', 'bener-bener', 'membantu', 'dan', 'melayani', 'dengan', 'baik', 'ketika', 'ada', 'kendala', '.', 'sukses', 'terus', 'yaa', 'qpon', '.']
Tokens : 20

--- QPon Review #4 ---
BEFORE : br mau login, malah gbisa nih.. ket nya akun terdeteksi tdk normal
AFTER : ['br', 'mau', 'login', ',', 'malah', 'gbisa', 'nih', '..', 'ket', 'nya', 'akun', 'terdeteksi', 'tdk', 'normal']
Tokens : 14

--- QPon Review #5 ---
BEFORE : apk fomo doang bisa sekarang susah banyak p

In [25]:
# ============================================
# STEMMING & STOPWORD REMOVAL \u2014 QPON DATA
# ============================================
# --- Stemming demo ---
long_words = [w for w in set(all_tokens_qpon) if len(w) > 6 and w.isalpha()]
sample_words = sorted(long_words, key=lambda w: freq_qpon[w], reverse=True)[:12]
print('=== STEMMING DEMO (QPON WORDS) ===')
print(f'{"Word":25s} {"Freq":>6s} -> Stem')
print('-' * 55)
for kata in sample_words:
 stem = stemmer.stem(kata)
 flag = ' *' if stem != kata else ''
 print(f'{kata:25s} {freq_qpon[kata]:>6,} -> {stem}{flag}')
print()
# --- Stopword removal demo ---
stop_id = set(stopwords.words('indonesian'))
print(f'Indonesian stopwords in NLTK: {len(stop_id)}')
print()
print('=== STOPWORD REMOVAL (3 QPON REVIEWS) ===')
for i in range(min(3, len(sample_reviews))):
 review = str(sample_reviews[i])
 tokens = word_tokenize(review.lower())
 filtered = [w for w in tokens if w not in stop_id and w.isalpha()]
 print(f'\n--- Review #{i+1} ---')
 print(f'BEFORE ({len(tokens)} tokens): {tokens}')
 print(f'AFTER ({len(filtered)} tokens): {filtered}')
 print(f'Removed: {len(tokens) - len(filtered)} stopwords / punctuation')

=== STEMMING DEMO (QPON WORDS) ===
Word                        Freq -> Stem
-------------------------------------------------------
aplikasi                   1,408 -> aplikasi
voucher                      794 -> voucher
makanan                      320 -> makan *
padahal                      291 -> padahal
membantu                     219 -> bantu *
gampang                      217 -> gampang
download                     184 -> download
dipakai                      183 -> pakai *
aplikasinya                  167 -> aplikasi *
restoran                     143 -> restoran
langsung                     125 -> langsung
terlalu                      118 -> terlalu

Indonesian stopwords in NLTK: 757

=== STOPWORD REMOVAL (3 QPON REVIEWS) ===

--- Review #1 ---
BEFORE (3 tokens): ['diskon', 'ya', 'gede']
AFTER (3 tokens): ['diskon', 'ya', 'gede']
Removed: 0 stopwords / punctuation

--- Review #2 ---
BEFORE (7 tokens): ['setiap', 'mau', 'pesen', 'tulisannya', 'terlalu', 'banyak', 'peminat']
AFT